# Taller 4 - Analisis exploratorio de Disney API

Este notebook lee desde MongoDB los datos crudos guardados por `ingesta.py`, selecciona variables relevantes y realiza un EDA con cinco insights y tres graficos.

# 1. Importar librerias y conectar con MongoDB

In [4]:
from pymongo import MongoClient
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "taller4_db"
COLLECTION_NAME = "raw_data"

client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]

total_documentos = collection.count_documents({})
print(f"Documentos encontrados en MongoDB: {total_documentos}")

Documentos encontrados en MongoDB: 100


# 2. Cargar datos crudos y seleccionar variables

Lectura de documentos crudos desde MongoDB y construcción de DataFrame con variables utiles para el analisis.

In [5]:
raw_data = list(collection.find())
df_raw = pd.DataFrame(raw_data)
df_raw.head()

,_id,films,shortFilms,tvShows,videoGames,parkAttractions,allies,enemies,name,imageUrl,url,alignment
0,112,[Hercules (film)],[],[Hercules (TV series)],[Kingdom Hearts III],[],[],[],Achilles,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/112,NaN
1,18,"[The Fox and the Hound, The Fox and the Hound 2]",[],[],[],[],[],[],Abigail the Cow,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/18,NaN
2,16,[Cheetah],[],[],[],[],[],[],Abdullah,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/16,NaN
3,45,"[Mary Poppins (film), Mary Poppins Returns]",[],[],[],[Disney Movie Magic],[],[],Admiral Boom and Mr. Binnacle,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/45,NaN
4,7,[],[],[Gravity Falls],[Disney Heroes: Battle Mode],[],[],[],.GIFfany,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/7,NaN


In [6]:
def contar_lista(valor):
    return len(valor) if isinstance(valor, list) else 0

df = pd.DataFrame({
    "name": df_raw["name"],
    "films_count": df_raw["films"].apply(contar_lista),
    "short_films_count": df_raw["shortFilms"].apply(contar_lista),
    "tv_shows_count": df_raw["tvShows"].apply(contar_lista),
    "video_games_count": df_raw["videoGames"].apply(contar_lista),
    "park_attractions_count": df_raw["parkAttractions"].apply(contar_lista),
    "allies_count": df_raw["allies"].apply(contar_lista),
    "enemies_count": df_raw["enemies"].apply(contar_lista),
    "has_image": df_raw["imageUrl"].notna()
})

columnas_apariciones = ["films_count", "short_films_count", "tv_shows_count", "video_games_count", "park_attractions_count"]
df["total_appearances"] = df[columnas_apariciones].sum(axis=1)
df["has_films"] = df["films_count"].apply(lambda x: "Con peliculas" if x > 0 else "Sin peliculas")

df.head()

,name,films_count,short_films_count,tv_shows_count,video_games_count,park_attractions_count,allies_count,enemies_count,has_image,total_appearances,has_films
0,Achilles,1,0,1,1,0,0,0,True,3,Con peliculas
1,Abigail the Cow,2,0,0,0,0,0,0,True,2,Con peliculas
2,Abdullah,1,0,0,0,0,0,0,True,1,Con peliculas
3,Admiral Boom and Mr. Binnacle,2,0,0,0,1,0,0,True,3,Con peliculas
4,.GIFfany,0,0,1,1,0,0,0,True,2,Sin peliculas


# 3. Inspección básica
Revisión de primeras filas, tipos de datos y valores nulos.

In [11]:
df.head(10)

,name,films_count,short_films_count,tv_shows_count,video_games_count,park_attractions_count,allies_count,enemies_count,has_image,total_appearances,has_films
0,Achilles,1,0,1,1,0,0,0,True,3,Con peliculas
1,Abigail the Cow,2,0,0,0,0,0,0,True,2,Con peliculas
2,Abdullah,1,0,0,0,0,0,0,True,1,Con peliculas
3,Admiral Boom and Mr. Binnacle,2,0,0,0,1,0,0,True,3,Con peliculas
4,.GIFfany,0,0,1,1,0,0,0,True,2,Sin peliculas
5,90's Adventure Bear,0,0,1,0,0,0,0,True,1,Sin peliculas
6,Candace Adams,0,0,1,0,0,0,0,True,1,Sin peliculas
7,Ahadi,2,0,0,0,0,0,0,True,2,Con peliculas
8,Al Muddy Sultan,0,0,1,0,0,0,0,True,1,Sin peliculas
9,Irwina Allen,0,0,1,0,0,0,0,True,1,Sin peliculas


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   name                    100 non-null    str  
 1   films_count             100 non-null    int64
 2   short_films_count       100 non-null    int64
 3   tv_shows_count          100 non-null    int64
 4   video_games_count       100 non-null    int64
 5   park_attractions_count  100 non-null    int64
 6   allies_count            100 non-null    int64
 7   enemies_count           100 non-null    int64
 8   has_image               100 non-null    bool 
 9   total_appearances       100 non-null    int64
 10  has_films               100 non-null    str  
dtypes: bool(1), int64(8), str(2)
memory usage: 8.0 KB


In [14]:
df.isnull().sum()

name                      0
films_count               0
short_films_count         0
tv_shows_count            0
video_games_count         0
park_attractions_count    0
allies_count              0
enemies_count             0
has_image                 0
total_appearances         0
has_films                 0
dtype: int64

# 4. Insights númericos
Cálculo de cinco datos relevantes sobre los personajes analizados.

In [22]:
total_personajes = len(df)
promedio_peliculas = df["films_count"].mean()
personaje_mas_videojuegos = df.loc[df["video_games_count"].idxmax(), ["name", "video_games_count"]]
personaje_mas_apariciones = df.loc[df["total_appearances"].idxmax(), ["name", "total_appearances"]]
personajes_sin_peliculas = (df["films_count"] == 0).sum()
personaje_mas_peliculas = df.loc[df["films_count"].idxmax(), ["name", "films_count"]]
totales_por_medio = df[columnas_apariciones].sum().sort_values(ascending=False)

print(f"Total de personajes analizados: {total_personajes}")
print(f"1. Promedio de peliculas por personaje: {promedio_peliculas:.2f}")
print(f"2. Personaje con más videojuegos registrados: {personaje_mas_videojuegos['name']} ({personaje_mas_videojuegos['video_games_count']} videojuegos)")
print(f"3. Personaje con más apariciones registradas: {personaje_mas_apariciones['name']} ({personaje_mas_apariciones['total_appearances']} apariciones)")
print(f"4. Personajes que no aparecen en ninguna película: {personajes_sin_peliculas}")
print(f"5. Personaje con más peliculas registradas: {personaje_mas_peliculas['name']} ({personaje_mas_peliculas['films_count']} peliculas)")
print("\nTotal de apariciones por tipo de medio:")
print(totales_por_medio)

Total de personajes analizados: 100
1. Promedio de peliculas por personaje: 0.57
2. Personaje con más videojuegos registrados: Baloo (13 videojuegos)
3. Personaje con más apariciones registradas: Baloo (37 apariciones)
4. Personajes que no aparecen en ninguna película: 61
5. Personaje con más peliculas registradas: B-Dawg (7 peliculas)

Total de apariciones por tipo de medio:
tv_shows_count            70
films_count               57
video_games_count         44
park_attractions_count    27
short_films_count          5
dtype: int64
